# Hybrid Legal Retrieval with Qwen3-Embedding & Qwen3-Reranker

This notebook implements a **hybrid retrieval pipeline** for the [LLM Agentic Legal Information Retrieval](https://www.kaggle.com/competitions/llm-agentic-legal-information-retrieval) competition.

## The Challenge
- **Cross-lingual**: English queries must retrieve German/French/Italian Swiss legal documents
- **Two corpora**: 175K law articles + 2.47M court decisions
- **Exact match**: Citation-level Macro F1 (string must match exactly)

## Key Improvements over Baseline

| # | Improvement | Baseline (BGE-M3) | This Notebook |
|---|-----------|-------------------|---------------|
| 1 | **Embedding model** | BGE-M3 | Qwen3-Embedding-0.6B (MTEB Retrieval 64.64 vs 54.60) |
| 2 | **Instruction prefix** | None | Task-specific instruction for queries |
| 3 | **Reranker** | BGE-Reranker-v2-M3 (CrossEncoder) | Qwen3-Reranker-0.6B (correct causal LM loading) |
| 4 | **Text composition** | `text` only | `citation + title + text` for laws |
| 5 | **Dense search** | NumPy dot product | FAISS IndexFlatIP |
| 6 | **Query expansion** | Top-1 law citation | Top-3 law citations |
| 7 | **Evaluation** | None | Val set Macro F1 |

### Important: Correct Qwen3-Reranker Usage

Qwen3-Reranker is a **causal language model**, not a sequence classification model.  
Loading it with `CrossEncoder` produces random scores (classification head is uninitialized).  
This notebook demonstrates the **correct approach**: use `AutoModelForCausalLM` and extract yes/no token probabilities.

## Model Selection: Why Qwen3?

### Embedding Models (0.6B class)

| Model | Size | MTEB Retrieval | Instruction Retrieval | STS |
|-------|------|---------------|----------------------|-----|
| BGE-M3 | 0.6B | 54.60 | -3.11 | 74.12 |
| multilingual-e5-large-instruct | 0.6B | 57.12 | -0.40 | 76.81 |
| **Qwen3-Embedding-0.6B** | **0.6B** | **64.64** | **+5.09** | **76.17** |

Qwen3-Embedding leads in retrieval (+10 pts over BGE-M3) and is the only model with **positive** instruction-following behavior.

### Reranker Models (< 1B)

| Model | Size | MTEB-R | MMTEB-R | FollowIR |
|-------|------|--------|---------|----------|
| BGE-reranker-v2-m3 | 0.6B | 57.03 | 58.36 | -0.01 |
| **Qwen3-Reranker-0.6B** | **0.6B** | **65.80** | **66.36** | **+5.41** |

Qwen3-Reranker outperforms BGE by +8.8 pts on MTEB-R and has strong instruction-following capabilities.

Source: [Qwen3-Embedding-0.6B HuggingFace](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B)

In [ ]:
!pip install -q sentence_transformers bm25s faiss-cpu

## Configuration

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings
import gc
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import bm25s
import faiss
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# --- Paths ---
TRAIN_PATH = '/kaggle/input/competitions/llm-agentic-legal-information-retrieval/train.csv'
VAL_PATH = '/kaggle/input/competitions/llm-agentic-legal-information-retrieval/val.csv'
TEST_PATH = '/kaggle/input/competitions/llm-agentic-legal-information-retrieval/test.csv'
LAWS_PATH = '/kaggle/input/competitions/llm-agentic-legal-information-retrieval/laws_de.csv'
COURT_PATH = '/kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv'
OUTPUT_PATH = 'submission.csv'

# --- Models ---
EMBEDDING_MODEL_NAME = 'Qwen/Qwen3-Embedding-0.6B'
RERANKER_MODEL_NAME = 'Qwen/Qwen3-Reranker-0.6B'

# --- Task instruction (used by both embedding and reranker) ---
TASK_INSTRUCTION = (
    'Given a legal question, retrieve relevant Swiss law articles and court decisions.'
)

# --- Hyperparameters ---
TOP_K_RETRIEVAL = 60       # candidates from BM25/FAISS before reranking
TOP_K_FINAL = 10           # final citations after reranking
TEXT_TRUNCATE = 3500       # max chars for reranker input
RRF_K = 60                 # reciprocal rank fusion parameter
LAWS_EXPANSION_TOP_K = 3   # law citations used for court query expansion
EMBEDDING_BATCH_SIZE = 16
RERANKER_BATCH_SIZE = 8

# --- GPU allocation ---
USE_GPU = torch.cuda.is_available()
NUM_GPUS = torch.cuda.device_count() if USE_GPU else 0

if NUM_GPUS >= 2:
    DEVICE_EMB = 'cuda:0'
    DEVICE_RERANK = 'cuda:1'
elif NUM_GPUS == 1:
    DEVICE_EMB = 'cuda:0'
    DEVICE_RERANK = 'cuda:0'
else:
    DEVICE_EMB = 'cpu'
    DEVICE_RERANK = 'cpu'

print(f'GPUs: {NUM_GPUS} | Embedding: {DEVICE_EMB} | Reranker: {DEVICE_RERANK}')

## Data Loading

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

laws_df = pd.read_csv(LAWS_PATH)
laws_df['text'] = laws_df['text'].fillna('')
laws_df['title'] = laws_df['title'].fillna('') if 'title' in laws_df.columns else ''

court_df = pd.read_csv(COURT_PATH, usecols=['citation', 'text'])
court_df['text'] = court_df['text'].fillna('')

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Laws: {len(laws_df)} | Court: {len(court_df)}')

## Text Composition

**Key improvement**: Concatenate `citation + title + text` for laws (baseline uses `text` only).  
Including the citation string and title in the embedding text improves retrieval of specific articles.

In [ ]:
# Laws: citation + title + text (richer representation)
laws_texts = (
    laws_df['citation'] + ' ' + laws_df['title'] + ' ' + laws_df['text']
).tolist()

# Court: citation + text
court_texts = (
    court_df['citation'] + ' ' + court_df['text']
).tolist()

# Lookup dicts for reranking
law_citation_to_text = dict(zip(laws_df['citation'], laws_texts))
court_citation_to_text = dict(zip(court_df['citation'], court_texts))

print(f'Laws text sample: {laws_texts[0][:120]}...')
print(f'Court text sample: {court_texts[0][:120]}...')

## BM25 Lexical Index

Classical keyword-based retrieval with German stopwords.  
Used for both laws and court corpora.

In [ ]:
def build_bm25_index(texts, name=''):
    truncated = [str(t).lower()[:TEXT_TRUNCATE] for t in texts]
    tokens = bm25s.tokenize(truncated, stopwords='de')
    retriever = bm25s.BM25()
    retriever.index(tokens)
    del tokens, truncated
    gc.collect()
    print(f'BM25 index ready: {name}')
    return retriever


def search_bm25(retriever, query, top_k):
    query_tokens = bm25s.tokenize([query.lower()], stopwords='de')
    docs, scores = retriever.retrieve(query_tokens, k=top_k)
    return [(int(docs[0][i]), float(scores[0][i])) for i in range(len(docs[0]))]


print('Building BM25 indexes...')
bm25_laws = build_bm25_index(laws_texts, 'laws')
bm25_courts = build_bm25_index(court_texts, 'court')

## Qwen3-Embedding: Instruction-Aware Encoding

This is the **most important difference** from other embedding models.

Qwen3-Embedding uses **instruction-aware prefixes**:
- **Query**: `"Instruct: {task_description}\nQuery: {query_text}"`
- **Document**: No prefix (encode as-is)

This tells the model *what kind of retrieval task* we're doing, significantly improving relevance for domain-specific queries.

```
# Example
query_input  = "Instruct: Given a legal question, retrieve relevant Swiss law articles...\nQuery: What are the consequences of breach of contract?"
doc_input    = "Art. 97 OR Der Schuldner hat dem Glaeuibiger den Schaden zu ersetzen..."  # no prefix
```

In [ ]:
print(f'Loading embedding model: {EMBEDDING_MODEL_NAME} on {DEVICE_EMB}')
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE_EMB,
    model_kwargs={'torch_dtype': torch.float16},
)
embedding_model.max_seq_length = 512
print('Embedding model ready')

In [ ]:
def encode_queries(model, queries):
    """Encode queries WITH instruction prefix (Qwen3 convention)."""
    prefixed = [
        f'Instruct: {TASK_INSTRUCTION}\nQuery: {q}'
        for q in queries
    ]
    return model.encode(
        prefixed,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    )


def encode_corpus(model, texts):
    """Encode documents WITHOUT prefix (Qwen3 convention)."""
    truncated = [str(t)[:TEXT_TRUNCATE] for t in texts]
    return model.encode(
        truncated,
        batch_size=EMBEDDING_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    )

## Laws Corpus Encoding + FAISS Index

Encode 175K law articles and build a FAISS inner-product index for fast dense retrieval.  
Court corpus (2.47M rows) is too large to encode in a notebook — we use BM25 only for court.

In [ ]:
print('Encoding laws corpus...')
start = time.time()
laws_embeddings = encode_corpus(embedding_model, laws_texts)
laws_embeddings = np.array(laws_embeddings, dtype=np.float32)
print(f'Laws encoding done: {laws_embeddings.shape} in {time.time()-start:.1f}s')

# Build FAISS index (inner product on normalized vectors = cosine similarity)
faiss_index = faiss.IndexFlatIP(laws_embeddings.shape[1])
faiss_index.add(laws_embeddings)
print(f'FAISS index ready: {faiss_index.ntotal} vectors, dim={laws_embeddings.shape[1]}')

# Free embedding array memory (index has its own copy)
del laws_embeddings
gc.collect()
torch.cuda.empty_cache()

## Qwen3-Reranker: Correct Loading with AutoModelForCausalLM

### Why Not CrossEncoder?

Qwen3-Reranker is a **causal language model** that judges relevance by generating "yes" or "no".  
Using `sentence_transformers.CrossEncoder` loads it as a sequence classification model, but the classification head  
is **randomly initialized** (not trained), producing meaningless scores.

### Correct Approach

1. Load with `AutoModelForCausalLM`
2. Format query-document pairs using the chat template
3. Get the logits for the last token
4. Extract probabilities for "yes" vs "no" tokens
5. Use P(yes) as the relevance score

```
<|im_start|>system
Judge whether the Document meets the requirements...<|im_end|>
<|im_start|>user
<Instruct>: Given a legal question, retrieve relevant Swiss law articles...
<Query>: What are the consequences of breach of contract?
<Document>: Art. 97 OR Der Schuldner hat dem Glaeuibiger...<|im_end|>
<|im_start|>assistant
<think>

</think>

-> model outputs logits -> P("yes") = relevance score
```

In [ ]:
class Qwen3Reranker:
    """Qwen3-Reranker-0.6B using the correct AutoModelForCausalLM approach."""

    SYSTEM_PROMPT = (
        'Judge whether the Document meets the requirements based on the '
        'Query and the Instruct provided. Note that the answer can only be '
        '"yes" or "no".'
    )

    def __init__(self, model_name, device='cuda', dtype=torch.float16):
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            padding_side='left',
            trust_remote_code=True,
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            trust_remote_code=True,
        ).to(device).eval()

        self.device = device
        self.yes_id = self.tokenizer.convert_tokens_to_ids('yes')
        self.no_id = self.tokenizer.convert_tokens_to_ids('no')

    def _format_pair(self, instruction, query, document):
        return (
            f'<|im_start|>system\n{self.SYSTEM_PROMPT}<|im_end|>\n'
            f'<|im_start|>user\n'
            f'<Instruct>: {instruction}\n'
            f'<Query>: {query}\n'
            f'<Document>: {document}\n'
            f'<|im_end|>\n'
            f'<|im_start|>assistant\n<think>\n\n</think>\n\n'
        )

    @torch.no_grad()
    def predict(self, pairs, batch_size=8, instruction=None):
        """Score query-document pairs. Returns list of relevance scores (0-1)."""
        if instruction is None:
            instruction = TASK_INSTRUCTION

        all_scores = []
        for start in range(0, len(pairs), batch_size):
            batch = pairs[start:start + batch_size]
            prompts = [
                self._format_pair(instruction, q, d[:TEXT_TRUNCATE])
                for q, d in batch
            ]
            inputs = self.tokenizer(
                prompts,
                padding=True,
                truncation=True,
                max_length=4096,
                return_tensors='pt',
            ).to(self.device)

            logits = self.model(**inputs).logits[:, -1, :]
            yes_no = torch.stack(
                [logits[:, self.no_id], logits[:, self.yes_id]], dim=1
            )
            probs = F.softmax(yes_no, dim=1)
            scores = probs[:, 1].cpu().numpy()  # P(yes)
            all_scores.extend(scores.tolist())

        return all_scores


print(f'Loading reranker: {RERANKER_MODEL_NAME} on {DEVICE_RERANK}')
reranker = Qwen3Reranker(RERANKER_MODEL_NAME, device=DEVICE_RERANK)
print('Reranker ready')

## Retrieval Functions

In [ ]:
def search_faiss(index, query_embedding, top_k):
    """Dense retrieval using FAISS inner-product index."""
    D, I = index.search(query_embedding.reshape(1, -1), top_k)
    return [(int(I[0][i]), float(D[0][i])) for i in range(len(I[0]))]


def reciprocal_rank_fusion(sparse_results, dense_results, k=RRF_K):
    """Merge BM25 and dense rankings using RRF: score = sum(1 / (k + rank))."""
    scores = {}
    for rank, (idx, _) in enumerate(sparse_results):
        scores[idx] = scores.get(idx, 0) + 1.0 / (k + rank)
    for rank, (idx, _) in enumerate(dense_results):
        scores[idx] = scores.get(idx, 0) + 1.0 / (k + rank)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [idx for idx, _ in ranked]


def rerank_candidates(query, candidates):
    """Rerank (citation, text) candidates using Qwen3-Reranker."""
    if not candidates:
        return []
    pairs = [[query, text] for _, text in candidates]
    scores = reranker.predict(pairs, batch_size=RERANKER_BATCH_SIZE)
    scored = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [(cit, sc) for (cit, _), sc in scored]

## Main Retrieval Pipeline

For each query:
1. **Laws**: Hybrid search (BM25 + FAISS + RRF)
2. **Query expansion**: Append top-3 law citations to query
3. **Court**: BM25 search with expanded query
4. **Merge & deduplicate** law + court candidates
5. **Rerank** all candidates with Qwen3-Reranker
6. **Return** top-K citations

In [ ]:
def retrieve_citations(query, query_embedding):
    """Full retrieval pipeline for a single query."""
    # 1. Laws: hybrid search (BM25 + FAISS + RRF)
    sparse_results = search_bm25(bm25_laws, query, TOP_K_RETRIEVAL)
    dense_results = search_faiss(faiss_index, query_embedding, TOP_K_RETRIEVAL)
    fused_indices = reciprocal_rank_fusion(sparse_results, dense_results)
    law_citations = [
        laws_df.iloc[idx]['citation']
        for idx in fused_indices[:TOP_K_RETRIEVAL]
        if 0 <= idx < len(laws_df)
    ]

    # 2. Query expansion with top-3 law citations
    expansion = ' '.join(law_citations[:LAWS_EXPANSION_TOP_K])
    expanded_query = f'{query} {expansion}' if expansion else query

    # 3. Court: BM25 search
    court_results = search_bm25(bm25_courts, expanded_query, TOP_K_RETRIEVAL)
    court_citations = [
        court_df.iloc[idx]['citation']
        for idx, _ in court_results
        if 0 <= idx < len(court_df)
    ]

    # 4. Merge & deduplicate
    seen = set()
    all_citations = []
    for cit in law_citations + court_citations:
        if cit not in seen:
            all_citations.append(cit)
            seen.add(cit)

    # 5. Prepare candidates for reranking
    candidates = []
    for cit in all_citations[:TOP_K_RETRIEVAL]:
        text = law_citation_to_text.get(cit) or court_citation_to_text.get(cit, '')
        if text:
            candidates.append((cit, text))

    # 6. Rerank
    reranked = rerank_candidates(query, candidates)
    return [cit for cit, _ in reranked[:TOP_K_FINAL]]

## Evaluation on Validation Set

Compute citation-level Macro F1 on the 10 validation queries.

In [ ]:
def compute_macro_f1(all_predicted, all_gold):
    """Citation-level Macro F1 (competition metric)."""
    f1_scores = []
    for pred, gold in zip(all_predicted, all_gold):
        pred_set = set(pred)
        gold_set = set(gold)
        if not gold_set:
            continue
        tp = len(pred_set & gold_set)
        precision = tp / len(pred_set) if pred_set else 0.0
        recall = tp / len(gold_set) if gold_set else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        f1_scores.append(f1)
    return np.mean(f1_scores) if f1_scores else 0.0


# Encode validation queries
val_queries = val_df['query'].tolist()
print(f'Encoding {len(val_queries)} validation queries...')
val_query_embeddings = encode_queries(embedding_model, val_queries)

# Run retrieval on validation set
print('Running validation...')
val_predictions = []
val_gold = []

for i, row in val_df.iterrows():
    pred = retrieve_citations(row['query'], val_query_embeddings[i])
    val_predictions.append(pred)
    gold = [c.strip() for c in str(row.get('gold_citations', '')).split(';') if c.strip()]
    val_gold.append(gold)

val_f1 = compute_macro_f1(val_predictions, val_gold)
print(f'\nValidation Macro F1: {val_f1:.5f}')
print(f'Baseline (BGE-M3):   0.02087')
print(f'Competition 1st:     0.27\n')

# Per-query breakdown
for i, (pred, gold) in enumerate(zip(val_predictions, val_gold)):
    pred_set, gold_set = set(pred), set(gold)
    tp = len(pred_set & gold_set)
    p = tp / len(pred_set) if pred_set else 0
    r = tp / len(gold_set) if gold_set else 0
    f = 2*p*r/(p+r) if (p+r) > 0 else 0
    print(f'  Query {i+1}: F1={f:.4f} | P={p:.4f} R={r:.4f} | pred={len(pred)} gold={len(gold)} tp={tp}')

## Generate Submission

In [ ]:
start_time = time.time()

# Batch-encode all test queries
test_queries = test_df['query'].tolist()
print(f'Encoding {len(test_queries)} test queries...')
test_query_embeddings = encode_queries(embedding_model, test_queries)

# Run retrieval
print('Retrieving citations...')
submission_data = []
for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Test queries'):
    citations = retrieve_citations(row['query'], test_query_embeddings[i])
    submission_data.append({
        'query_id': row['query_id'],
        'predicted_citations': ';'.join(citations),
    })

submission_df = pd.DataFrame(submission_data)
submission_df.to_csv(OUTPUT_PATH, index=False)

elapsed = time.time() - start_time
print(f'\nCompleted in {elapsed/60:.1f} minutes')
print(f'Saved {len(submission_df)} predictions to {OUTPUT_PATH}')
submission_df.head()

## Results & Discussion

### What This Notebook Demonstrates

1. **Correct Qwen3-Reranker loading** using `AutoModelForCausalLM` with yes/no probability extraction
2. **Instruction-aware encoding** with Qwen3-Embedding (significant for cross-lingual retrieval)
3. **Improved text composition** (citation + title + text) for better semantic matching
4. **FAISS** for efficient dense search at scale

### Further Improvements (Not Covered Here)

- **Dense encoding for court corpus**: Embedding 2.47M court decisions would improve court retrieval significantly
- **Fine-tuning**: Domain-adapted Qwen3-Embedding on Swiss legal data
- **Agentic RAG**: Multi-step retrieval with LLM-guided query reformulation
- **Metadata enrichment**: Adding English summaries and Q&A pairs to bridge the cross-lingual gap
- **Chunking + MaxSim**: Splitting long court texts into chunks for better matching

### References

- [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B)
- [Qwen3-Reranker-0.6B](https://huggingface.co/Qwen/Qwen3-Reranker-0.6B)
- [Qwen3-Embedding Blog](https://qwenlm.github.io/blog/qwen3-embedding/)